# Gaussian Mixture Models — Rock Decomposition
## Machine learning for energy applications — Module 3, Exercise 3

In this exercise, we will use **Gaussian Mixture Models (GMMs)** to cluster geochemical rock data. The dataset contains major-oxide weight percentages for various igneous rock samples. Each sample also has a reference rock-type label (`ROCK1`, e.g. KOMATIITE). The label is **never a model input**: the GMM is fitted in a fully **unsupervised** manner on the ten oxides only, and `ROCK1` is used afterwards to compare the components with the named rock types.

GMMs extend K-Means by modelling each cluster as a multivariate Gaussian distribution, which allows:
- **Soft (probabilistic) assignments** — each sample has a probability of belonging to every cluster.
- **Elliptical cluster shapes** — clusters are not restricted to spherical geometry.
- **Model selection via BIC/AIC** — principled way to choose the number of components.

**Topics covered:**
1. Load and explore the dataset
2. Missing values, feature selection and standardization
3. Fit a GMM with an initial guess of 3 components
4. Hard cluster assignments and PCA visualization
5. Soft assignments — probability heatmap
6. Model selection with BIC and AIC
7. Refit with optimal number of components
8. Compare GMM clusters with actual rock types
9. Explore covariance types
10. Summary

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['figure.dpi'] = 100

---
## 1. Load and Explore the Dataset

Load `rock_multiclass_raw.csv` and inspect its shape, data types, missing values, and the distribution of rock types.

**Tasks:**
- Load the CSV into a DataFrame
- Print the shape, column names, data types, and missing value counts
- Display the first few rows

In [ ]:
# YOUR CODE HERE
# Hint: Use pd.read_csv('rock_multiclass_raw.csv')
# Print df.shape, df.columns, df.dtypes, df.isnull().sum()


**Tasks:**
- Print the value counts of the `ROCK1` column
- Create a bar chart showing the distribution of rock types

In [ ]:
# YOUR CODE HERE
# Hint: df['ROCK1'].value_counts() and .plot(kind='bar')


In [ ]:
# YOUR CODE HERE
# Hint: df.describe() for summary statistics


---
## 2. Feature Selection and Standardization

Select only the numeric geochemical features (exclude the `ROCK1` label column) and standardize them using `StandardScaler`.

**Missing values first.** `GaussianMixture` does not accept NaN, and nine of the ten oxide columns have gaps. The provided cell below drops every row with a missing oxide, plus any row without a `ROCK1` label (needed for the comparison in Section 8), and prints which columns caused the loss and how the rock-type counts change. Imputation is the alternative, but it would invent compositions.

**Exploratory analysis of one fixed dataset.** The scaler is fitted on all remaining rows because we only describe these samples; nothing is predicted for new samples, so there is no training/test split.

**Tasks:**
- Run the provided missing-value cell and note how many rows remain
- Extract the feature matrix `X` (the ten oxides in `feature_cols`) and the rock labels
- Fit a `StandardScaler` and transform `X`

In [ ]:
# Rows with a missing oxide cannot be used (GaussianMixture rejects NaN), and the
# comparison in Section 8 needs a reference label, so drop both kinds of row.
feature_cols = [c for c in df.columns if c != 'ROCK1']
n_before = len(df)
n_missing = df[feature_cols + ['ROCK1']].isna().sum()
print("Missing values per column (a row can miss several):")
print(n_missing[n_missing > 0].sort_values(ascending=False).to_string())

counts_before = df['ROCK1'].value_counts()
df = df.dropna(subset=feature_cols + ['ROCK1']).reset_index(drop=True)
print(f"\nRows kept: {len(df)} of {n_before} ({n_before - len(df)} dropped)")
print(pd.DataFrame({'before': counts_before, 'after': df['ROCK1'].value_counts()})
      .sort_values('before', ascending=False))

In [ ]:
# YOUR CODE HERE
# Hint: feature_cols (the ten oxides) was defined in the missing-value cell above
# X = df[feature_cols].to_numpy()
# rock_labels = df['ROCK1'].to_numpy()
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)


In [ ]:
# YOUR CODE HERE
# Hint: Print X_scaled.mean(axis=0) and X_scaled.std(axis=0) to verify


---
## 3. Fit a GMM with 3 Components (Initial Guess)

As a starting point, fit a `GaussianMixture` with `n_components=3`. We will refine this later using BIC/AIC.

**Tasks:**
- Create a `GaussianMixture(n_components=3, random_state=42, n_init=5)`
- Fit it on `X_scaled`
- Print convergence status, BIC, AIC, and cluster weights

In [ ]:
# YOUR CODE HERE
# Hint: gmm3 = GaussianMixture(n_components=3, random_state=42, n_init=5)
# gmm3.fit(X_scaled)
# Check: gmm3.converged_, gmm3.bic(X_scaled), gmm3.aic(X_scaled), gmm3.weights_


---
## 4. Hard Assignment — PCA 2D Projection

Use `.predict()` to assign each sample to its most likely cluster. Visualize with a 2D PCA projection.

**Tasks:**
- Use `gmm3.predict(X_scaled)` to get hard cluster labels
- Print cluster sizes
- Fit PCA with 2 components on `X_scaled`
- Create a scatter plot of PC1 vs PC2, coloured by GMM cluster

In [ ]:
# YOUR CODE HERE
# Hint: labels_gmm3 = gmm3.predict(X_scaled)
# Use np.unique(labels_gmm3, return_counts=True) for cluster sizes


In [ ]:
# YOUR CODE HERE
# Hint: pca = PCA(n_components=2)
# X_pca = pca.fit_transform(X_scaled)
# plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_gmm3, cmap='viridis', ...)


---
## 5. Soft Assignment — Probability Heatmap

Use `.predict_proba()` to obtain the probability that each sample belongs to each cluster. Visualize as a heatmap.

**Tasks:**
- Call `gmm3.predict_proba(X_scaled)` to get the probability matrix
- Display the first 5 rows as a DataFrame
- Sort samples by hard assignment and create a heatmap with `sns.heatmap()`

In [ ]:
# YOUR CODE HERE
# Hint: probs = gmm3.predict_proba(X_scaled)
# pd.DataFrame(probs, columns=[f'Cluster {i}' for i in range(3)]).head()


In [ ]:
# YOUR CODE HERE
# Hint: sort_idx = np.argsort(labels_gmm3)
# probs_sorted = probs[sort_idx]
# sns.heatmap(probs_sorted, cmap='YlOrRd', ...)


---
## 6. Model Selection — BIC and AIC

Fit GMMs with `n_components` from 1 to 10 and compute the **BIC** (Bayesian Information Criterion) and **AIC** (Akaike Information Criterion) for each. Lower values indicate a better model.

**Tasks:**
- Loop over `n_components` in `range(1, 11)`
- For each, fit a GMM and store `.bic(X_scaled)` and `.aic(X_scaled)`
- Plot both curves on the same axes
- Identify the optimal K for each criterion

In [ ]:
# YOUR CODE HERE
# Hint:
# n_range = range(1, 11)
# bics, aics = [], []
# for n in n_range:
#     gmm_temp = GaussianMixture(n_components=n, random_state=42, n_init=5)
#     gmm_temp.fit(X_scaled)
#     bics.append(gmm_temp.bic(X_scaled))
#     aics.append(gmm_temp.aic(X_scaled))


In [ ]:
# YOUR CODE HERE
# Hint: Plot bics and aics vs n_range using plt.plot()
# Use np.argmin(bics) and np.argmin(aics) to find optimal K


---
## 7. Refit with Optimal Number of Components

Choose the optimal number of components based on the BIC curve and refit the model.

**Tasks:**
- Set `optimal_k` to the BIC-optimal value
- Refit a GMM with `n_init=10` for stability
- Print cluster weights and sizes
- Visualize in PCA space

In [ ]:
# YOUR CODE HERE
# Hint: optimal_k = best_bic_n
# gmm_opt = GaussianMixture(n_components=optimal_k, random_state=42, n_init=10)
# gmm_opt.fit(X_scaled)
# labels_opt = gmm_opt.predict(X_scaled)


In [ ]:
# YOUR CODE HERE
# Hint: Scatter plot of X_pca coloured by labels_opt


---
## 8. Compare GMM Clusters with Reference Rock Types

`ROCK1` was not used to fit the model, so we can now compare the components with the reference rock types, using a **cross-tabulation** and the **Adjusted Rand Index (ARI)**. The names are a reference, not a ground truth for oxide structure: some are defined partly by other criteria (gabbro is the coarse-grained intrusive equivalent of basalt; tholeiite is a basalt series), so only partial agreement is expected. Treat each component as a hypothesis about a population in oxide space, not as a rock type.

**Tasks:**
- Create a cross-tabulation with `pd.crosstab(rock_labels, labels_opt)`
- Plot it as a heatmap
- Compute the Adjusted Rand Index using `adjusted_rand_score` (print the label encoding if you encode the names)
- Compute the silhouette score
- Create a side-by-side PCA plot comparing the reference labels vs GMM clusters

In [ ]:
# YOUR CODE HERE
# Hint: ct = pd.crosstab(rock_labels, labels_opt)
# sns.heatmap(ct, annot=True, fmt='d', cmap='Blues')


In [ ]:
# YOUR CODE HERE
# Hint: from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# rock_encoded = le.fit_transform(rock_labels)
# print({name: code for code, name in enumerate(le.classes_)})  # the encoding
# ari = adjusted_rand_score(rock_encoded, labels_opt)
# sil = silhouette_score(X_scaled, labels_opt)


In [ ]:
# YOUR CODE HERE
# Hint: fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# Left: scatter X_pca coloured by reference rock type (loop over unique rock_labels)
# Right: scatter X_pca coloured by labels_opt


---
## 9. Explore Covariance Types

The `covariance_type` parameter controls the shape of the Gaussian clusters:
- `'full'` — each cluster has its own general covariance matrix (most flexible)
- `'tied'` — all clusters share the same covariance matrix
- `'diag'` — each cluster has its own diagonal covariance (axis-aligned ellipses)
- `'spherical'` — each cluster has a single variance (spherical clusters, like K-Means)

**Tasks:**
- Loop over the four covariance types
- For each, fit a GMM with `optimal_k` components and compute BIC, AIC, silhouette, and ARI
- Collect results in a DataFrame
- Create a 2x2 subplot grid visualizing the clusters for each covariance type

In [ ]:
# YOUR CODE HERE
# Hint:
# cov_types = ['full', 'tied', 'diag', 'spherical']
# for cov_type in cov_types:
#     gmm_cov = GaussianMixture(n_components=optimal_k, covariance_type=cov_type,
#                                random_state=42, n_init=5)
#     gmm_cov.fit(X_scaled)
#     ... compute BIC, AIC, silhouette_score, adjusted_rand_score ...


In [ ]:
# YOUR CODE HERE
# Hint: fig, axes = plt.subplots(2, 2, figsize=(14, 12))
# for ax, cov_type in zip(axes.ravel(), cov_types):
#     ... fit GMM, predict, scatter plot on ax ...


---
## 10. Summary

In this exercise we:

1. **Loaded and explored** the rock geochemistry dataset containing major-oxide weight percentages for multiple igneous rock types.
2. **Dropped rows with missing values** (printing which columns caused the loss and how many rows remain), **standardized** the ten oxides, and kept the rock-type label out of the model inputs.
3. **Fitted an initial GMM** with K=3 components and visualized hard cluster assignments in PCA space.
4. **Examined soft assignments** using `predict_proba()` and a probability heatmap, showing how GMMs provide richer information than hard clustering.
5. **Performed model selection** using BIC and AIC curves across K=1..10, choosing K by the lowest BIC.
6. **Refitted the model** with the BIC-chosen K and compared the components with the reference rock types via cross-tabulation and Adjusted Rand Index.
7. **Explored covariance types** (`full`, `tied`, `diag`, `spherical`) and compared their BIC, silhouette, and ARI scores.

**Key takeaways:**
- GMMs provide probabilistic (soft) cluster assignments, unlike K-Means which gives only hard assignments.
- BIC and AIC balance fit quality against model complexity. BIC chooses the number of components that best describes the oxide distribution, not the number of rock types.
- The `covariance_type` parameter has a significant impact on cluster shapes and model flexibility.
- Some components line up with compositional end-members while others split or merge named rock types. A component is a hypothesis about structure in the oxide space; naming it a rock type needs further evidence.